Import bibliothèque

In [59]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import re

Chargement de donnees

In [60]:

RAW_DATA_DIR = os.path.join("..","data", "raw")

In [61]:
df_auth = pd.read_csv(os.path.join(RAW_DATA_DIR, 'authentication_logs.csv'))
df_edr = pd.read_csv(os.path.join(RAW_DATA_DIR, 'edr_alerts.csv'))
df_assets = pd.read_csv(os.path.join(RAW_DATA_DIR, 'assets.csv'))
df_users = pd.read_csv(os.path.join(RAW_DATA_DIR, 'users.csv'))

In [62]:
df_auth.head()

,event_id,timestamp,user_id,src_ip,device_id,event_type,authentication_result,location,severity
0,AUTH002391,2026-03-10 02:31:16,U0247,10.14.245.152,D0098,VPN_CONNECT,success,Lille,LOW
1,AUTH001438,2026-03-19 13:53:30,U0090,10.20.221.161,D0030,MFA_CHALLENGE,Success,Nantes,low
2,AUTH001801,10/03/2026 09:19,U0331,10.42.70.40,D0004,PASSWORD_CHANGE,SUCCESS,Nantes,MEDIUM
3,AUTH002344,2026-03-09 11:04:58,U0199,10.23.20.8,D0008,LOGIN,SUCCESS,Bordeaux,medium
4,AUTH000264,2026-03-05 15:14:01,U0030,10.32.221.253,D0131,PASSWORD_CHANGE,SUCCESS,Lyon,Low


In [63]:
df_edr.head()

,alert_id,timestamp,device_id,user_id,alert_type,process_name,severity,status,analyst_decision
0,EDR001382,2026-03-11 10:03:00,D0162,U0131,Suspicious Process,backup_agent.exe,HIGH,Closed,FALSE_POSITIVE
1,EDR000843,2026-03-18 14:20:42,D0088,U0019,Unusual Network,outlook.exe,Medium,Open,FALSE_POSITIVE
2,EDR001258,2026-03-11 03:51:00,D0015,U0066,Suspicious Process,backup_agent.exe,High,Closed,FALSE_POSITIVE
3,EDR001222,2026-03-11 02:03:00,D0094,U0030,Suspicious Process,backup_agent.exe,HIGH,Closed,FALSE_POSITIVE
4,EDR000109,2026-03-12 15:18:33,D0185,U0041,PUP Detected,outlook.exe,Low,In Progress,NaN


In [64]:
df_assets.head()

,device_id,hostname,asset_type,department,owner,criticality,operating_system
0,D0035,PHF-SRV-0035,Mobile,Regions,Juridique,Medium,Windows Server 2019
1,D0075,PHF-VM-0075,Server,Regions,Regions,Critical,Windows 11
2,D0051,PHF-SRV-0051,VM,Regions,Regions,Critical,macOS 14
3,D0137,PHF-WS-0137,Mobile,Juridique,Regions,Critical,Windows 10
4,D0078,PHF-LT-0078,Desktop,Communication,Communication,MEDIUM,Windows 10


In [65]:
df_users.head()

,user_id,department,role,employment_status,privileged_account,location
0,U0026,RH,Director,ACTIVE,NO,Lyon
1,U0312,RH,Analyst,ACTIVE,NO,Nantes
2,U0248,Direction,Engineer,ACTIVE,NO,Toulouse
3,U0287,Communication,Engineer,ACTIVE,NO,Lille
4,U0341,Direction,Assistant,ACTIVE,NO,Toulouse


### 1. COMPLETENESS (Complétude / Valeurs manquantes)


In [66]:
print("\n COMPLETENESS (Valeurs manquantes)")

print("*" * 50)

datasets = {
    'authentication_logs.csv': df_auth,
    'edr_alerts.csv': df_edr,
    'assets.csv': df_assets,
    'users.csv': df_users
}

for nom, df in datasets.items():
    missing = df.isnull().sum()
    missing_crit = missing[missing > 0]
    if len(missing_crit) > 0:
        print(f" {nom} :")
        for col, val in missing_crit.items():
            pct = (val / len(df)) * 100
            print(f"   - {col} : {val} manquantes ({pct:.1f}%)")
            print("-" * 50)
    else:
        print(f" {nom} : Aucune valeur manquante.")


 COMPLETENESS (Valeurs manquantes)
**************************************************
 authentication_logs.csv :
   - user_id : 151 manquantes (3.0%)
--------------------------------------------------
 edr_alerts.csv :
   - analyst_decision : 234 manquantes (12.1%)
--------------------------------------------------
 assets.csv :
   - owner : 14 manquantes (7.0%)
--------------------------------------------------
   - criticality : 20 manquantes (10.0%)
--------------------------------------------------
 users.csv : Aucune valeur manquante.


### 2. UNIQUENESS (Unicité / Doublons)

In [69]:
print("\nUNIQUENESS (Doublons exacts et clés primaires) ")

print("*" * 50)

pk_mapping = {
    'authentication_logs.csv': ('event_id', df_auth),
    'edr_alerts.csv': ('alert_id', df_edr),
    'assets.csv': ('device_id', df_assets),
    'users.csv': ('user_id', df_users)
}

for nom, (id, df) in pk_mapping.items():
    critiques = df.duplicated().sum()
    dups_pk = df[id].duplicated().sum() if id in df.columns else 0
    print(f" {nom} -> Doublons exacts : {critiques} | Doublons sur la clé '{id}' : {dups_pk}")
    print("-" * 50)



UNIQUENESS (Doublons exacts et clés primaires) 
**************************************************
 authentication_logs.csv -> Doublons exacts : 145 | Doublons sur la clé 'event_id' : 145
--------------------------------------------------
 edr_alerts.csv -> Doublons exacts : 74 | Doublons sur la clé 'alert_id' : 74
--------------------------------------------------
 assets.csv -> Doublons exacts : 0 | Doublons sur la clé 'device_id' : 0
--------------------------------------------------
 users.csv -> Doublons exacts : 5 | Doublons sur la clé 'user_id' : 5
--------------------------------------------------


### 4. VALIDITY (Validité des formats et structures)

In [68]:
print("\n VALIDITY (Validité technique des champs)")

print("*" * 50)
# Vérification des adresses IP dans authentication_logs
if 'src_ip' in df_auth.columns:
    ipv4_regex = re.compile(r'^(?:[0-9]{1,3}\.){3}[0-9]{1,3}$')
    invalid_ips = df_auth['src_ip'].astype(str).apply(lambda x: not bool(ipv4_regex.match(x))).sum()
    print(f" authentication_logs.csv :")
    print(f"   • Adresses IP mal formées ou invalides : {invalid_ips} / {len(df_auth)}")

# Vérification des formats de dates (timestamps)
for name, df in [("authentication_logs.csv", df_auth), ("edr_alerts.csv", df_edr)]:
    if 'timestamp' in df.columns:
        parsed_dates = pd.to_datetime(df['timestamp'], errors='coerce')
        invalid_dates = parsed_dates.isnull().sum()
        print(f" {name} :")
        print(f"   • Timestamp non convertibles (invalides) : {invalid_dates}")
        print("-" * 50)


 VALIDITY (Validité technique des champs)
**************************************************
 authentication_logs.csv :
   • Adresses IP mal formées ou invalides : 32 / 4992
 authentication_logs.csv :
   • Timestamp non convertibles (invalides) : 3322
--------------------------------------------------
 edr_alerts.csv :
   • Timestamp non convertibles (invalides) : 482
--------------------------------------------------


In [37]:
df_auth.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4992 entries, 0 to 4991
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   event_id               4992 non-null   object
 1   timestamp              4992 non-null   object
 2   user_id                4841 non-null   object
 3   src_ip                 4992 non-null   object
 4   device_id              4992 non-null   object
 5   event_type             4992 non-null   object
 6   authentication_result  4992 non-null   object
 7   location               4992 non-null   object
 8   severity               4992 non-null   object
dtypes: object(9)
memory usage: 351.1+ KB


In [40]:
df_edr.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1929 entries, 0 to 1928
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   alert_id          1929 non-null   object
 1   timestamp         1929 non-null   object
 2   device_id         1929 non-null   object
 3   user_id           1929 non-null   object
 4   alert_type        1929 non-null   object
 5   process_name      1929 non-null   object
 6   severity          1929 non-null   object
 7   status            1929 non-null   object
 8   analyst_decision  1695 non-null   object
dtypes: object(9)
memory usage: 135.8+ KB


### 5. CONSISTENCY (Cohérence textuelle et des catégories)

In [55]:
print("\n CONSISTENCY (Cohérence des catégories)")

print("*" * 50)
if 'operating_system' in df_assets.columns:
    os_uniques = df_assets['operating_system'].unique()
    print(f"assets.csv - Variabilité de la casse/dénomination (OS) : {len(os_uniques)} variantes détectées")
    print(f"   • Échantillon : {list(os_uniques)[:5]}")
    print("-" * 50)

if 'event_type' in df_auth.columns:
    events = df_auth['event_type'].unique()
    print(f"authentication_logs.csv - Types d'événements uniques : {list(events)}")
    print("-" * 50)


 CONSISTENCY (Cohérence des catégories)
**************************************************
assets.csv - Variabilité de la casse/dénomination (OS) : 6 variantes détectées
   • Échantillon : ['Windows Server 2019', 'Windows 11', 'macOS 14', 'Windows 10', 'Windows Server 2022']
--------------------------------------------------
authentication_logs.csv - Types d'événements uniques : ['VPN_CONNECT', 'MFA_CHALLENGE', 'PASSWORD_CHANGE', 'LOGIN', 'RDP_SESSION', 'LOGOUT']
--------------------------------------------------


### 6. ACCURACY & TIMELINESS (Exactitude et Actualité des référentiels)

In [58]:
print("\nACCURACY & TIMELINESS (Intégrité et Actualité inter-fichiers)")

print("*" * 50)
# Croisement pour détecter le Shadow IT ou les données obsolètes (Timeliness / Consistency)
auth_users = set(df_auth['user_id'].dropna())
ref_users = set(df_users['user_id'].dropna())
orphaned_users = auth_users - ref_users

auth_devs = set(df_auth['device_id'].dropna())
ref_devs = set(df_assets['device_id'].dropna())
orphaned_devs = auth_devs - ref_devs

print(f"   • Utilisateurs dans les logs absents de users.csv : {len(orphaned_users)} (Risque d'obsolescence/IAM)")
print(f"   • Machines (devices) dans les logs absentes de assets.csv : {len(orphaned_devs)} (Risque Shadow IT)")



ACCURACY & TIMELINESS (Intégrité et Actualité inter-fichiers)
**************************************************
   • Utilisateurs dans les logs absents de users.csv : 6 (Risque d'obsolescence/IAM)
   • Machines (devices) dans les logs absentes de assets.csv : 8 (Risque Shadow IT)
